# Workshop AI Summit - IA Multimodal con Snowflake Cortex

**Duración estimada:** 20 minutos | **Idioma:** Español

En este Workshop aprenderás a procesar **imágenes, documentos y audio** con funciones de IA nativas de Snowflake, y a construir un **agente conversacional** usando lenguaje natural con **Cortex Code**.

## Qué vas a hacer
1. Analizar imágenes con AI_COMPLETE multimodal
2. Extraer datos estructurados de contratos PDF/DOCX con AI_PARSE_DOCUMENT y AI_EXTRACT
3. Transcribir y analizar sentimiento de llamadas con AI_TRANSCRIBE y AI_SENTIMENT
4. Crear un agente conversacional desde cero usando **Cortex Code en Snowsight**

> Asegúrate de tener seleccionado el warehouse `HOL_WH` y el schema `HOL_AI_SUMMIT.PUBLIC` arriba a la derecha del notebook.

## Ejercicio 1: Imágenes con IA Multimodal

Snowflake puede analizar imágenes directamente con AI_COMPLETE y modelos como Claude. Vamos a describir un siniestro vehicular y extraer datos de una cédula.

In [ ]:
-- Que imágenes tenemos en el stage?
SELECT RELATIVE_PATH, SIZE FROM DIRECTORY(@IMAGENES) ORDER BY RELATIVE_PATH;

In [ ]:
-- Analizar imagen del siniestro vehicular con IA multimodal
SELECT AI_COMPLETE(
  'claude-4-sonnet',
  'Eres un perito de seguros. Describe el dano del vehiculo en esta imagen, indica severidad (leve/moderado/grave) y estima un rango de costo de reparacion en USD. Responde en español y en formato JSON con campos: descripcion, severidad, costo_estimado.',
  TO_FILE('@IMAGENES', 'choque.png')
) AS analisis_siniestro;

In [ ]:
-- Extraer datos de identificacion de una cédula (KYC)
SELECT AI_COMPLETE(
  'claude-4-sonnet',
  'Extrae los datos de esta cédula en formato JSON: numero_documento, nombres, apellidos, fecha_nacimiento, lugar_expedicion. Si no puedes leer un campo, devuelve null.',
  TO_FILE('@IMAGENES', 'cédula.jpg')
) AS datos_cédula;

## Ejercicio 2: Documentos con AI_PARSE_DOCUMENT y AI_EXTRACT

Snowflake parsea DOCX, PDF, PPTX y más formatos sin pre-procesamiento. Después podemos extraer campos estructurados con AI_EXTRACT.

In [ ]:
-- Ver los contratos ya parseados (creados por setup.sql)
SELECT file_name, LEFT(content, 300) AS preview FROM DOCS_PARSED;

In [ ]:
-- Extraer campos estructurados de los contratos de arrendamiento
SELECT
  file_name,
  AI_EXTRACT(
    content,
    ['arrendador', 'arrendatario', 'valor_canon_mensual', 'plazo_meses', 'direccion_inmueble', 'fecha_inicio']
  ) AS campos_extraidos
FROM DOCS_PARSED;

In [ ]:
-- Generar un resumen ejecutivo de cada contrato
SELECT
  file_name,
  AI_SUMMARIZE_AGG(content) AS resumen
FROM DOCS_PARSED
GROUP BY file_name;

## Ejercicio 3: Audio con AI_TRANSCRIBE, AI_SENTIMENT y AI_COMPLETE

Transcribimos llamadas de servicio al cliente, analizamos el sentimiento con **`AI_SENTIMENT`** y generamos recomendaciones para el asesor con **`AI_COMPLETE`**.

**Separación clara de responsabilidades:**
- `AI_SENTIMENT` → análisis estructurado de sentimiento (general y por aspectos)
- `AI_CLASSIFY` → categorización del motivo de la llamada
- `AI_COMPLETE` → recomendaciones y coaching personalizado al asesor

In [ ]:
-- Ver transcripciones (generadas por setup.sql con AI_TRANSCRIBE)
SELECT file_name, LEFT(transcripcion, 250) AS preview
FROM TRANSCRIPCIONES;

In [ ]:
-- Análisis de sentimiento con AI_SENTIMENT (categorías y polaridad)
SELECT
  file_name,
  AI_SENTIMENT(transcripcion):categories[0]:sentiment::VARCHAR AS sentimiento_general,
  AI_SENTIMENT(
    transcripcion,
    ['servicio al cliente', 'producto', 'precio', 'asesor', 'tiempo de respuesta']
  ) AS sentimiento_por_aspecto
FROM TRANSCRIPCIONES;

In [ ]:
-- Recomendaciones para el asesor con AI_COMPLETE (LLM coaching)
SELECT
  file_name,
  AI_COMPLETE(
    'claude-4-sonnet',
    CONCAT(
      'Eres un coach experto en servicio al cliente. Analiza esta transcripción y genera recomendaciones para el asesor en formato JSON con: ',
      '{"puntos_dolor": [...], "fortalezas_asesor": [...], "areas_mejora": [...], "recomendacion_inmediata": "...", "script_sugerido": "frase exacta a usar en la próxima interacción"}. ',
      'Responde SOLO el JSON, en español. Transcripción: ', transcripcion
    )
  ) AS recomendaciones_asesor
FROM TRANSCRIPCIONES;

In [ ]:
-- Scorecard 360: sentimiento (AI_SENTIMENT) + categoría (AI_CLASSIFY) + nota del asesor (AI_COMPLETE)
SELECT
  file_name AS llamada,
  AI_SENTIMENT(transcripcion):categories[0]:sentiment::VARCHAR AS sentimiento,
  AI_CLASSIFY(
    transcripcion,
    ['Reclamo', 'Consulta', 'Oferta comercial', 'Soporte técnico', 'Otro']
  ):labels[0]::VARCHAR AS categoria,
  AI_COMPLETE(
    'claude-4-sonnet',
    CONCAT(
      'Califica del 1 al 10 al asesor en empatía, resolución y profesionalismo. ',
      'Responde SOLO un JSON: {"empatia": N, "resolucion": N, "profesionalismo": N, "nota_general": N, "veredicto": "texto corto"}. ',
      'Transcripción: ', transcripcion
    )
  ) AS evaluacion_asesor
FROM TRANSCRIPCIONES;

## Ejercicio 4: Cortex Code - Crea tu agente con lenguaje natural

Este es el momento estrella del Workshop. Vamos a salir del notebook y usar **Cortex Code** dentro de Snowsight para construir piezas con IA escribiendo solo en español.

### Pasos
1. En Snowsight, abre **Cortex Code** (icono en el panel lateral o atajo Cmd/Ctrl+I).
2. Asegúrate de estar en el contexto: HOL_AI_SUMMIT.PUBLIC con warehouse HOL_WH.
3. Copia y pega cada uno de estos prompts (uno a la vez) y deja que Cortex Code genere el SQL/código:

**Prompt 1 - Vista de imágenes clasificadas:**
> Crea una vista llamada V_IMAGENES_CLASIFICADAS que recorra todos los archivos del stage IMAGENES y agregue una columna con la clasificación del tipo de imagen (cédula, accidente vehicular, factura, otro) usando AI_CLASSIFY sobre AI_COMPLETE multimodal.

**Prompt 2 - Tabla unificada multimodal:**
> Crea una vista llamada V_HOL_360 que combine las filas de DOCS_PARSED y TRANSCRIPCIONES en un solo dataset con columnas: tipo_fuente, archivo, contenido, sentimiento.

**Prompt 3 - Probar el agente:**
> Genera un SELECT que invoque a AI_COMPLETE con un prompt al agente AGENTE_HOL preguntándole: cuánto es el canon mensual del contrato de arrendamiento numero 2 y cuál fue el sentimiento de la última llamada?

**Prompt 4 - App de Streamlit:**
> Crea una app Streamlit in Snowflake llamada DASHBOARD_HOL que muestre: total de imágenes por tipo, sentimiento de las llamadas en gráfico de barras, y un campo de chat conectado al agente AGENTE_HOL.

Estos prompts demuestran como **una persona sin conocimientos técnicos avanzados** puede crear pipelines de IA de extremo a extremo en Snowflake en cuestión de minutos.

## Ejercicio 5: Cortex Analyst + Cortex Search + Snowflake Intelligence

Ahora vamos a crear **desde cero** los tres componentes clave de Snowflake Intelligence usando **Cortex Code**.
El objetivo es que el agente pueda responder preguntas combinando datos estructurados (pólizas, clientes, siniestros) con documentos no estructurados (contratos, llamadas).

### Arquitectura que vamos a construir
```
┌─────────────────────────────────────────────────────────────┐
│           SNOWFLAKE INTELLIGENCE (Agente)                   │
├──────────────────────┬──────────────────────────────────────┤
│  Cortex Analyst      │  Cortex Search                       │
│  (datos estructurados)│  (documentos no estructurados)      │
│                      │                                      │
│  • Pólizas vendidas  │  • Contratos de arrendamiento        │
│  • Clientes          │  • Transcripciones de llamadas       │
│  • Reclamaciones     │                                      │
└──────────────────────┴──────────────────────────────────────┘
```

### Paso 1: Explorar los datos estructurados que tenemos

In [ ]:
-- Explorar las tablas de negocio (creadas por setup.sql)
SELECT 'POLIZAS' AS tabla, COUNT(*) AS filas FROM POLIZAS
UNION ALL SELECT 'CLIENTES', COUNT(*) FROM CLIENTES
UNION ALL SELECT 'RECLAMACIONES', COUNT(*) FROM RECLAMACIONES
UNION ALL SELECT 'BASE_CONOCIMIENTO', COUNT(*) FROM BASE_CONOCIMIENTO;

In [ ]:
-- Vista rápida de pólizas vendidas
SELECT fecha, tipo_poliza, producto, region, cliente, vendedor, prima_mensual, estado
FROM POLIZAS ORDER BY fecha DESC LIMIT 10;

### Paso 2: Crear Cortex Analyst con Cortex Code

Abre **Cortex Code** (`Cmd/Ctrl + I`) y pega este prompt:

> **Prompt - Crear Semantic View:**
> Crea una semantic view llamada SV_SEGUROS_DEMO sobre las tablas POLIZAS, CLIENTES y RECLAMACIONES en HOL_AI_SUMMIT.PUBLIC. Incluye dimensiones para tipo_poliza, región, vendedor, segmento de cliente y tipo de siniestro. Define métricas para total de primas, número de pólizas, monto total de reclamaciones aprobadas y tasa de aprobación. Agrega time_dimensions sobre las fechas y relaciones entre las tablas usando el campo cliente/nombre. Incluye verified_queries para: top vendedores por primas, reclamaciones pendientes, y distribución de pólizas por región.

**¿Qué es Cortex Analyst?** Es la capacidad de Snowflake que convierte preguntas en lenguaje natural a SQL usando un modelo semántico. La Semantic View le dice al modelo qué significan tus datos y cómo consultarlos correctamente.

Una vez creada, puedes probarla directamente:

In [ ]:
-- La Semantic View SV_SEGUROS ya fue creada por setup.sql
-- Verificar que existe:
SHOW SEMANTIC VIEWS IN HOL_AI_SUMMIT.PUBLIC;

### Paso 3: Crear Cortex Search con Cortex Code

Pega este prompt en **Cortex Code**:

> **Prompt - Crear Cortex Search Service:**
> Crea un Cortex Search Service llamado SEARCH_UNIFICADO sobre la tabla BASE_CONOCIMIENTO en HOL_AI_SUMMIT.PUBLIC. La columna de búsqueda principal es 'contenido', los atributos son 'tipo_documento' y 'file_name'. Usa el warehouse HOL_WH, target_lag de 1 hora, y el embedding model snowflake-arctic-embed-l-v2.0.

**¿Qué es Cortex Search?** Es un servicio de búsqueda semántica nativo de Snowflake que permite encontrar información relevante en documentos sin necesidad de una base de datos vectorial externa. Aquí indexamos contratos y transcripciones de llamadas.

Prueba buscando algo:

In [ ]:
-- Probar Cortex Search (el servicio DOCS_SEARCH ya existe del setup)
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'HOL_AI_SUMMIT.PUBLIC.DOCS_SEARCH',
  '{
    "query": "obligaciones del arrendatario",
    "columns": ["contenido", "tipo_documento", "file_name"],
    "limit": 2
  }'
) AS resultado;

### Paso 4: Crear el Agente con Snowflake Intelligence usando Cortex Code

Este es el momento culminante. Pega en **Cortex Code**:

> **Prompt - Crear Agente para Snowflake Intelligence:**
> Crea un agente llamado AGENTE_SEGUROS_DEMO en HOL_AI_SUMMIT.PUBLIC que combine:
> 1. Una herramienta cortex_analyst_text_to_sql conectada a la semantic view SV_SEGUROS para consultar datos de pólizas, clientes y reclamaciones
> 2. Una herramienta cortex_search conectada al servicio DOCS_SEARCH para buscar en contratos y transcripciones de llamadas
> 3. Una herramienta data_to_chart para generar gráficos
>
> El agente debe responder en español, es un asistente de una empresa de seguros en Colombia. Las instrucciones de orquestación deben indicar cuándo usar cada herramienta. Incluye 5 sample_questions sobre ventas, contratos, siniestros y sentimiento de llamadas.

**¿Qué es Snowflake Intelligence?** Es la capa conversacional donde un agente usa múltiples herramientas (Analyst + Search + Charts) para responder cualquier tipo de pregunta sobre tus datos, estructurados y no estructurados, en una sola interfaz.

### Paso 5: Probar el Agente en Snowflake Intelligence

1. Ve a **AI & ML > Snowflake Intelligence** en Snowsight
2. Selecciona **Agente Seguros 360** (creado por setup.sql)
3. Haz estas preguntas de prueba:

| Pregunta | Herramienta que usa | Tipo de información |
|----------|--------------------|-----------------------|
| ¿Cuál es el total de primas por región? | Cortex Analyst | Datos estructurados |
| ¿Qué dice el contrato sobre las cláusulas de terminación? | Cortex Search | Documentos |
| Muéstrame un gráfico de reclamaciones por tipo de siniestro | Analyst + Chart | Datos + Visual |
| ¿Qué ofrecieron en la última llamada telefónica? | Cortex Search | Audio transcrito |
| ¿Quién es el mejor vendedor y cuántas reclamaciones tiene? | Analyst (multi-tabla) | Cruce de datos |

**El agente decide automáticamente qué herramienta usar** según la pregunta. Esto es el poder de Snowflake Intelligence: una sola interfaz conversacional que combina TODOS tus datos.

## Cierre - Lo que aprendiste

| Capacidad | Función / Herramienta | Diferenciador |
|-----------|----------------------|---------------|
| Análisis de imágenes | AI_COMPLETE multimodal | Sin GPU, sin APIs externas |
| Parsing de documentos | AI_PARSE_DOCUMENT | Soporta PDF, DOCX, PPTX nativo |
| Extracción estructurada | AI_EXTRACT | JSON listo para BI |
| Transcripción + sentimiento | AI_TRANSCRIBE + AI_SENTIMENT | Audio multilingüe |
| Búsqueda semántica | CORTEX SEARCH SERVICE | Sin vector DB externa |
| Text-to-SQL inteligente | CORTEX ANALYST + Semantic View | Preguntas en lenguaje natural → SQL |
| Agentes conversacionales | SNOWFLAKE INTELLIGENCE | Combina datos estructurados + documentos |
| Generación de gráficos | data_to_chart | Visualización automática |
| Creación sin código | Cortex Code | Todo construido con prompts en español |

### Mensaje clave para tu organización

**Snowflake es la única plataforma donde puedes:**
1. Procesar imágenes, documentos y audio sin mover datos
2. Crear modelos semánticos que entienden tu negocio
3. Indexar documentos con búsqueda semántica nativa
4. Construir agentes inteligentes que combinan TODO
5. Hacer todo lo anterior con lenguaje natural (Cortex Code)

**Una sola plataforma gobernada. Una experiencia conversacional. Resultados en minutos.**